In [2]:
from google.colab import drive
import os

# Mount your Google Drive
drive.mount('/content/drive')

# Set base paths
DATASET_PATH = '/content/drive/MyDrive/archive'
IMAGES_PATH = os.path.join(DATASET_PATH, 'Images')
CAPTIONS_FILE = os.path.join(DATASET_PATH, 'captions.txt')

# Check files
print("📁 Contents of dataset folder:")
print(os.listdir(DATASET_PATH))

print("\n📂 Image folder:", IMAGES_PATH)
print("📄 Captions file:", CAPTIONS_FILE)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📁 Contents of dataset folder:
['captions.txt', '.DS_Store', 'Images']

📂 Image folder: /content/drive/MyDrive/archive/Images
📄 Captions file: /content/drive/MyDrive/archive/captions.txt


In [3]:
import os
import re
import string
import random
from tensorflow.keras.preprocessing.text import Tokenizer

# PATHS
DATASET_PATH = '/content/drive/MyDrive/archive'
CAPTIONS_FILE = os.path.join(DATASET_PATH, 'captions.txt')

# STEP 1: Load and clean captions
def load_clean_captions(filepath):
    captions = {}
    with open(filepath, 'r') as file:
        next(file)
        for line in file:
            if line.strip() == "":
                continue
            img, caption = line.strip().split(',', 1)
            caption = caption.lower()
            caption = caption.translate(str.maketrans('', '', string.punctuation))
            caption = re.sub(r'\d+', '', caption)
            caption = re.sub(r'\s+', ' ', caption).strip()
            if img not in captions:
                captions[img] = []
            captions[img].append(caption)
    return captions

captions = load_clean_captions(CAPTIONS_FILE)

# STEP 2: Add <startseq> and <endseq>
for img in captions:
    captions[img] = ['startseq ' + c + ' endseq' for c in captions[img]]

# STEP 3: Tokenizer, vocab size, max length
all_captions = [c for caplist in captions.values() for c in caplist]
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(c.split()) for c in all_captions)

# STEP 4: Dataset split (60/20/20)
image_ids = [img.split('.')[0] for img in captions]
random.seed(42)
random.shuffle(image_ids)
n = len(image_ids)

train_ids = image_ids[:int(0.6 * n)]
val_ids = image_ids[int(0.6 * n):int(0.8 * n)]
test_ids = image_ids[int(0.8 * n):]

def filter_captions(ids):
    return {img_id: captions[img_id + '.jpg'] for img_id in ids if img_id + '.jpg' in captions}

train_captions = filter_captions(train_ids)
val_captions = filter_captions(val_ids)
test_captions = filter_captions(test_ids)

print(f"✅ Ready:")
print(f" - Train: {len(train_captions)}")
print(f" - Val: {len(val_captions)}")
print(f" - Test: {len(test_captions)}")
print(f" - Vocab size: {vocab_size}, Max length: {max_length}")

✅ Ready:
 - Train: 4854
 - Val: 1618
 - Test: 1619
 - Vocab size: 8781, Max length: 37


In [4]:
from tensorflow.keras.preprocessing.text import Tokenizer

# Add tokens
for img_id in captions:
    captions[img_id] = ['startseq ' + c + ' endseq' for c in captions[img_id]]

# Flatten all captions
all_captions = [c for clist in captions.values() for c in clist]

# Tokenizer
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)

# Vocab size and max length
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(c.split()) for c in all_captions)

print(f"✅ Vocab size: {vocab_size}, Max caption length: {max_length}")

✅ Vocab size: 8781, Max caption length: 39


In [6]:
from tensorflow.keras.applications.inception_v3 import InceptionV3, preprocess_input
from tensorflow.keras.preprocessing.image import load_img, img_to_array
from tensorflow.keras.models import Model
from tqdm import tqdm
import numpy as np
import os
import pickle

# Load InceptionV3 without top layer
base_model = InceptionV3(weights='imagenet')
model = Model(inputs=base_model.input, outputs=base_model.get_layer('avg_pool').output)

# Extract features
def extract_features_for_ids(image_ids, images_path, model):
    features = {}
    for img_id in tqdm(image_ids):
        img_path = os.path.join(images_path, img_id + '.jpg')  # Correct now
        if not os.path.exists(img_path):
            continue
        img = load_img(img_path, target_size=(299, 299))
        img = img_to_array(img)
        img = np.expand_dims(img, axis=0)
        img = preprocess_input(img)
        feature = model.predict(img, verbose=0)
        features[img_id] = feature.flatten()
    return features

# Run extraction
features_train = extract_features_for_ids(train_ids, IMAGES_PATH, model)
features_val = extract_features_for_ids(val_ids, IMAGES_PATH, model)
features_test = extract_features_for_ids(test_ids, IMAGES_PATH, model)

# Save features
with open('features_train.pkl', 'wb') as f: pickle.dump(features_train, f)
with open('features_val.pkl', 'wb') as f: pickle.dump(features_val, f)
with open('features_test.pkl', 'wb') as f: pickle.dump(features_test, f)

print("✅ Features extracted and saved successfully:")
print(f" - Training: {len(features_train)}")
print(f" - Validation: {len(features_val)}")
print(f" - Test: {len(features_test)}")

100%|██████████| 1619/1619 [19:46<00:00,  1.36it/s]


✅ Features extracted and saved successfully:
 - Training: 4854
 - Validation: 1618
 - Test: 1619


In [5]:
import os, re, string, random
from tensorflow.keras.preprocessing.text import Tokenizer

# Path
DATASET_PATH = '/content/drive/MyDrive/archive'
CAPTIONS_FILE = os.path.join(DATASET_PATH, 'captions.txt')

# Load and clean captions
def load_clean_captions(path):
    captions = {}
    with open(path, 'r') as f:
        next(f)
        for line in f:
            if ',' not in line: continue
            img, caption = line.strip().split(',', 1)
            caption = caption.lower()
            caption = caption.translate(str.maketrans('', '', string.punctuation))
            caption = re.sub(r'\d+', '', caption)
            caption = re.sub(r'\s+', ' ', caption).strip()
            if img not in captions:
                captions[img] = []
            captions[img].append(caption)
    return captions

captions = load_clean_captions(CAPTIONS_FILE)

# Add <startseq> and <endseq>
for img in captions:
    captions[img] = ['startseq ' + c + ' endseq' for c in captions[img]]

# Tokenizer and vocab info
all_captions = [c for caplist in captions.values() for c in caplist]
tokenizer = Tokenizer()
tokenizer.fit_on_texts(all_captions)
vocab_size = len(tokenizer.word_index) + 1
max_length = max(len(c.split()) for c in all_captions)

# Dataset split
image_ids = [img.split('.')[0] for img in captions]
random.seed(42)
random.shuffle(image_ids)
n = len(image_ids)
train_ids = image_ids[:int(0.6 * n)]
val_ids = image_ids[int(0.6 * n):int(0.8 * n)]
test_ids = image_ids[int(0.8 * n):]

def filter_captions(ids):
    return {img_id: captions[img_id + '.jpg'] for img_id in ids if img_id + '.jpg' in captions}

train_captions = filter_captions(train_ids)
val_captions = filter_captions(val_ids)
test_captions = filter_captions(test_ids)

print("✅ Reset Complete:")
print(f" - Train: {len(train_captions)}, Val: {len(val_captions)}, Test: {len(test_captions)}")
print(f" - Vocab size: {vocab_size}, Max caption length: {max_length}")

✅ Reset Complete:
 - Train: 4854, Val: 1618, Test: 1619
 - Vocab size: 8781, Max caption length: 37


In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
import numpy as np
import pickle

# 1. Load pre-extracted training features
with open('features_train.pkl', 'rb') as f:
    features_train = pickle.load(f)

# 2. Keep only captions for images with extracted features
aligned_train_captions = {img_id: train_captions[img_id] for img_id in features_train.keys()}

# 3. Create input-output training sequences
def create_sequences(tokenizer, max_length, captions_dict, image_features, vocab_size):
    X1, X2, y = [], [], []
    for img_id, captions in captions_dict.items():
        feature = image_features[img_id]
        for caption in captions:
            seq = tokenizer.texts_to_sequences([caption])[0]
            for i in range(1, len(seq)):
                in_seq = seq[:i]
                out_seq = seq[i]
                in_seq = pad_sequences([in_seq], maxlen=max_length)[0]
                out_seq = to_categorical([out_seq], num_classes=vocab_size)[0]
                X1.append(feature)
                X2.append(in_seq)
                y.append(out_seq)
    return np.array(X1), np.array(X2), np.array(y)

# 4. Generate full training data
X1_train, X2_train, y_train = create_sequences(
    tokenizer=tokenizer,
    max_length=max_length,
    captions_dict=aligned_train_captions,
    image_features=features_train,
    vocab_size=vocab_size
)

# 5. Confirm result
print("✅ Sequence generation complete:")
print(f"🖼 X1 (image features): {X1_train.shape}")
print(f"📝 X2 (token inputs): {X2_train.shape}")
print(f"🎯 y (one-hot targets): {y_train.shape}")

In [1]:
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add
from tensorflow.keras.models import Model

# Image feature input (2048-dimensional from InceptionV3)
inputs1 = Input(shape=(2048,))
fe1 = Dropout(0.5)(inputs1)
fe2 = Dense(256, activation='relu')(fe1)

# Caption input sequence
inputs2 = Input(shape=(max_length,))
se1 = Embedding(vocab_size, 256, mask_zero=True)(inputs2)
se2 = Dropout(0.5)(se1)
se3 = LSTM(256)(se2)

# Merge image + caption branches
decoder1 = add([fe2, se3])
decoder2 = Dense(256, activation='relu')(decoder1)
outputs = Dense(vocab_size, activation='softmax')(decoder2)

# Final model
model = Model(inputs=[inputs1, inputs2], outputs=outputs)
model.compile(loss='categorical_crossentropy', optimizer='adam')

# Summary
model.summary()

NameError: name 'max_length' is not defined

In [ ]:
# Optional: if memory is limited, reduce batch size to 32
model.fit(
    [X1_train, X2_train],
    y_train,
    epochs=10,
    batch_size=64,
    verbose=1
)